# Deep Learning 027 — Activation Functions, Part 1

Companion notebook to the lesson. Without a non-linearity a stack of layers collapses to a
single layer, so the activation is what makes depth mean anything. Which one you pick then
decides how well gradients survive the trip back.

Three properties decide everything, and all three are measurable:

| | range | max derivative | zero-centred? | cost |
|---|---|---|---|---|
| **sigmoid** | (0, 1) | **0.25** | no | `exp` |
| **tanh** | (−1, 1) | **1.00** | **yes** | 2 × `exp` |
| **ReLU** | [0, ∞) | **1** (exactly) | no | one comparison |

`numpy` only; the plot cell is optional.

In [ ]:
import numpy as np
import time

def sigmoid(z): return 1 / (1 + np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)
def tanh(z):    return np.tanh(z)
def d_tanh(z):  return 1 - np.tanh(z) ** 2
def relu(z):    return np.maximum(0, z)
def d_relu(z):  return (z > 0) * 1.0

FUNCS = [("sigmoid", sigmoid, d_sigmoid), ("tanh", tanh, d_tanh), ("relu", relu, d_relu)]

## Part A — Why a non-linearity at all

Two linear layers are one linear layer. This is not an approximation — it is matrix
multiplication being associative.

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(5, 4))
W1, W2 = rng.normal(size=(4, 6)), rng.normal(size=(6, 3))

two_layers = (X @ W1) @ W2
one_layer = X @ (W1 @ W2)                 # W1 @ W2 is just some other 4x3 matrix
print("max difference:", np.abs(two_layers - one_layer).max())
assert np.allclose(two_layers, one_layer)
print("\nA 100-layer linear network has exactly the expressive power of one linear layer.")
print("The activation is what stops the collapse.")

## Part B — The derivative is the whole story for training

The chain rule multiplies one derivative per layer. Whatever the *largest* that derivative
can be sets a ceiling on how much gradient can survive a deep stack.

In [ ]:
z = np.linspace(-15, 15, 60001)
print(f"{'':>10}{'range of output':>22}{'max derivative':>18}{'at z =':>10}")
for name, f, df in FUNCS:
    d = df(z)
    lo, hi = f(z).min(), f(z).max()
    print(f"{name:>10}{f'({lo:.2f}, {hi:.2f})':>22}{d.max():>18.4f}{z[d.argmax()]:>10.2f}")

In [ ]:
print("what a chain of layers does to the gradient, best case:\n")
print(f"{'layers':>8}{'sigmoid':>14}{'tanh':>12}{'relu':>12}")
for n in (1, 5, 10, 20, 50):
    print(f"{n:>8}{0.25 ** n:>14.2e}{1.0 ** n:>12.2e}{1.0 ** n:>12.2e}")
print("\ntanh's ceiling is 1.00 - FOUR TIMES sigmoid's 0.25 - which is why tanh was")
print("the default for years before ReLU. But 'best case' is doing a lot of work here.")

## Part C — Saturation, which is where the best case stops applying

The ceiling above is reached only at `z = 0`. Move away from zero and both sigmoid and tanh
**flatten**, and a flat function has no gradient at all.

In [ ]:
print(f"{'z':>6}{'d sigmoid':>13}{'d tanh':>12}{'d relu':>10}")
for zi in (0, 1, 2, 4, 6, 10):
    print(f"{zi:>6}{d_sigmoid(zi):>13.6f}{d_tanh(zi):>12.6f}{d_relu(zi):>10.1f}")
print("\nAt z = 6, sigmoid and tanh have already given up. ReLU has not - its")
print("derivative is exactly 1 no matter how large z gets, which is the property")
print("that makes deep stacks trainable.")

In [ ]:
# how much of a realistic pre-activation range is 'usable'?
r = np.random.default_rng(1)
zs = r.normal(scale=3.0, size=200_000)          # typical spread in a trained layer
print(f"{'':>10}{'mean derivative':>18}{'saturated (0 < d < 0.01)':>28}{'exactly 0':>12}")
for name, f, df in FUNCS:
    d = df(zs)
    sat = ((d > 0) & (d < 0.01)).mean()
    print(f"{name:>10}{d.mean():>18.4f}{sat:>27.1%}{(d == 0).mean():>12.1%}")

The **mean derivative** column is the comparison: 0.115 for sigmoid, 0.256 for tanh, 0.499
for ReLU. Over a realistic spread of pre-activations, sigmoid passes back about a tenth of
whatever arrives, per layer, and tanh about a quarter.

The two right-hand columns separate two different failures that are easy to confuse.
**Saturation** is a derivative that is small but non-zero — tanh has 31.7% of its units
there, sigmoid 12.6%. **Exactly zero** is ReLU's, on 50.1% of units, and it is not
saturation at all: those units are simply switched off, and a unit that is off for *every*
example never comes back. That is the dying ReLU problem, and lesson 028 is about it.

So the argument for ReLU as the hidden-layer default is narrower than "it has no gradient
problem". It is that **wherever ReLU is active its derivative is exactly 1**, so an active
path through a deep stack loses nothing — while every sigmoid path loses a factor at every
layer whether it is saturated or not.

## Part D — Zero-centring, the property nobody notices

Sigmoid outputs are all positive. That means every input to the next layer is positive,
which constrains the *direction* the gradient can point for that layer's weights: they must
all increase together or all decrease together.

In [ ]:
r = np.random.default_rng(2)
z = r.normal(size=(4000, 8))
for name, f, _ in FUNCS:
    a = f(z)
    print(f"{name:>10}  mean output {a.mean():>8.4f}   share positive {(a > 0).mean():>6.1%}")
print("\nOnly tanh is centred on zero. Sigmoid's outputs average 0.5, so the next")
print("layer receives a large constant offset it has to learn to subtract.")

In [ ]:
# the consequence: with all-positive inputs, every weight in a neuron shares
# the sign of the incoming delta, so the update can only move diagonally
delta = np.array([0.7])                          # some upstream error signal
for name, f, _ in FUNCS:
    a = f(r.normal(size=(1, 6)))
    grad = a.T * delta                           # dL/dw for one neuron
    same_sign = np.abs(np.sign(grad).sum()) == len(grad)
    print(f"{name:>10}  gradient signs {np.sign(grad).ravel().astype(int)}"
          f"   all the same: {same_sign}")
print("\nAll-positive activations mean the weight vector zig-zags toward its target")
print("instead of heading straight for it. Tanh removes that constraint; ReLU does")
print("not, which is one of the few places tanh still wins.")

## Part E — Cost

The one property that is pure engineering. `exp` is expensive; `max(0, z)` is a comparison.

In [ ]:
big = np.random.default_rng(3).normal(size=(2000, 2000))

def timeit(f, reps=5):
    f(big)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); f(big); ts.append(time.perf_counter() - t0)
    return min(ts)

times = {name: timeit(f) for name, f, _ in FUNCS}
base = times["relu"]
print(f"{'':>10}{'ms':>10}{'vs relu':>10}")
for name, _, _ in FUNCS:
    print(f"{name:>10}{times[name] * 1e3:>10.1f}{times[name] / base:>9.1f}x")
print("\nOn 4 million elements, per forward pass, per layer, per epoch. It adds up.")

In [ ]:
# Optional plot. Skip if matplotlib is unavailable.
import matplotlib.pyplot as plt

zz = np.linspace(-6, 6, 500)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for name, f, df in FUNCS:
    ax[0].plot(zz, f(zz), label=name)
    ax[1].plot(zz, df(zz), label=name)
ax[0].set(title="activation", xlabel="z"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set(title="derivative", xlabel="z"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].axhline(0.25, ls=":", c="grey")
plt.tight_layout(); plt.show()

## Choosing

| Layer | Use | Why |
|---|---|---|
| hidden | **ReLU** | derivative exactly 1 where it is active, and cheap |
| hidden, if ReLU is dying | Leaky ReLU / ELU | lesson 028 |
| output, binary | sigmoid | you need a probability in (0, 1) |
| output, multi-class | softmax | you need probabilities that compete |
| output, regression | linear | the answer is a real number |

Note the asymmetry: **sigmoid is a bad hidden activation and the correct output activation
for binary classification.** Being saturating and bounded is a flaw in the middle of a
network and exactly the requirement at the end of one.

Lesson 028 covers ReLU's own failure mode — the derivative that is *exactly* zero on the
negative side — and the variants that fix it.

## Try it yourself

1. Compute the derivative ceiling for `softplus`, `log(1 + exp(z))`. Where does it sit
   relative to the three above?
2. Measure the `share of units with d < 0.01` in Part C for a pre-activation spread of
   `scale=1` instead of 3. How much of the sigmoid problem is really an initialisation
   problem?
3. Train the same small network with each activation and record how many epochs each needs
   to reach 90% training accuracy.
4. Show numerically that `tanh(z) = 2·sigmoid(2z) - 1`. Given that, why do they behave so
   differently in training?